# 7.0 Reports (run in SageMaker Studio)

One SageMaker Clarify job producing the **SHAP explainability** report only (no bias facet), then a consolidated index `monitoring/reports/index.json` of the data-quality and model-quality constraints/violations, the Clarify outputs and the Debugger report.

**Cost/risk note:** Clarify explainability starts a *shadow endpoint* inside its own processing job and deletes it when the job ends; no endpoint is created by this notebook. The job uses `ml.m5.large` x1 on a 200-row test sample. SHAP on test windows from seen subjects is still optimistic.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json, time
import boto3, numpy as np, pandas as pd
from botocore.exceptions import ClientError
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
from src.pamap2 import monitoring as M
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
cw = session.client("cloudwatch")
sm = session.client("sagemaker")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")
MON = lambda key: s3_uri(cfg, "monitoring", key)
NS = cfg["monitoring_namespace"]

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_bytes(uri):
    b, k = split_key(uri)
    return s3.get_object(Bucket=b, Key=k)["Body"].read()
def read_bytes_or_none(uri):
    try:
        return read_bytes(uri)
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise
def read_parquet(name):
    return pd.read_parquet(io.BytesIO(read_bytes(SPLITS + name)))
def read_json(uri):
    return json.loads(read_bytes(uri))
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)
def put_text(uri, text):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=text.encode())
    return uri
def exists(uri):
    b, k = split_key(uri)
    try:
        s3.head_object(Bucket=b, Key=k)
        return True
    except ClientError:
        return False
def latest_subprefix(uri_prefix):
    """Newest child 'folder' (timestamped run folders sort lexicographically)."""
    b, k = split_key(uri_prefix)
    r = s3.list_objects_v2(Bucket=b, Prefix=k, Delimiter="/")
    subs = sorted(p["Prefix"] for p in r.get("CommonPrefixes", []))
    return f"s3://{b}/{subs[-1]}" if subs else None
def publish(metrics):
    """metrics: {name: value} -> custom namespace (put_metric_data)."""
    cw.put_metric_data(Namespace=NS, MetricData=[{"MetricName": n, "Value": float(v), "Unit": "None"} for n, v in metrics.items()])
    print("Published", metrics)

def safe_publish(metrics):
    try:
        publish(metrics)
    except Exception as e:   # never mask the original error
        print("WARNING: publish failed:", e)
def report_failure(job_name):
    try:
        print("FailureReason:", sm.describe_processing_job(ProcessingJobName=job_name).get("FailureReason"))
    except Exception as e:
        print("Could not describe job:", e)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
ACTIVITY_IDS = list(LABEL_MAP.values())
print(K, "classes;", len(FEATURES), "features")


In [ ]:
import sagemaker
from sagemaker import clarify
sm_session = sagemaker.Session(boto_session=session)
train_info = read_json(REPORTS + "xgboost_training.json")
PKG_ARN = read_json(REPORTS + "xgboost_eval.json")["model_package_arn"]   # the evaluated package
CLARIFY_OUT = MON("reports/clarify/")


## Clarify SHAP explainability (billable: one ml.m5.large job)

In [ ]:
test = read_parquet("test.parquet").reset_index(drop=True)
sample = test.sample(n=min(200, len(test)), random_state=0)
medians = read_parquet("train.parquet")[FEATURES].median().fillna(0)
clarify_in = put_text(MON("reports/clarify/input/test_sample.csv"),
                      sample[FEATURES].fillna(medians).to_csv(header=False, index=False))   # no empty CSV fields
baseline_row = medians.tolist()
model_name = f"{cfg['prefix']}-mon-clarify-{int(time.time())}"
sm.create_model(ModelName=model_name, ExecutionRoleArn=cfg["role_arn"],
                Containers=[{"ModelPackageName": PKG_ARN}])
try:
    processor = clarify.SageMakerClarifyProcessor(role=cfg["role_arn"], instance_count=1, instance_type="ml.m5.large",
                                                  sagemaker_session=sm_session)
    data_config = clarify.DataConfig(s3_data_input_path=clarify_in, s3_output_path=CLARIFY_OUT + time.strftime("%Y%m%d-%H%M%S") + "/",
                                     headers=FEATURES, dataset_type="text/csv")
    model_config = clarify.ModelConfig(model_name=model_name, instance_type="ml.m5.large", instance_count=1,
                                       accept_type="text/csv", content_type="text/csv")
    label_cfg = clarify.ModelPredictedLabelConfig(label_headers=[ACTIVITY_NAMES[LABEL_MAP[i]] for i in range(K)])   # multiclass softprob
    shap = clarify.SHAPConfig(baseline=[baseline_row], num_samples=100, agg_method="mean_abs", save_local_shap_values=False)
    processor.run_explainability(data_config=data_config, model_config=model_config, explainability_config=shap,
                                 model_scores=label_cfg, wait=True, logs=False)
finally:
    sm.delete_model(ModelName=model_name)   # temporary model only
print("Clarify output:", data_config.s3_output_path)


## Consolidated report index

In [ ]:
dq, mq = latest_subprefix(MON("data-quality/runs/")), latest_subprefix(MON("model-quality/runs/"))
clarify_run = latest_subprefix(CLARIFY_OUT)
debugger = train_info.get("debugger_artifacts")
cands = {
    "data_quality_baseline_statistics": MON("data-quality/baseline/statistics.json"),
    "data_quality_baseline_constraints": MON("data-quality/baseline/constraints.json"),
    "data_quality_violations": dq and dq + "constraint_violations.json",
    "data_quality_violations_summary": dq and dq + "violations_summary.json",
    "model_quality_baseline_statistics": MON("model-quality/baseline/statistics.json"),
    "model_quality_baseline_constraints": MON("model-quality/baseline/constraints.json"),
    "model_quality_violations": mq and mq + "constraint_violations.json",
    "model_quality_violations_summary": mq and mq + "violations_summary.json",
    "clarify_analysis": clarify_run and clarify_run + "analysis.json",
    "clarify_report_html": clarify_run and clarify_run + "report.html",
    "debugger_report": debugger and debugger + M.DEBUGGER_REPORT_PATH,
}
# a violations file legitimately absent means zero violations; keep the summary as the evidence
cands = {k: (v if v and exists(v) else None) for k, v in cands.items()}
index = M.report_index(cands)
write_json(MON("reports/index.json"), index)
display(pd.DataFrame([{"report": k, "uri": v, "exists": exists(v)} for k, v in index["reports"].items()]))
print("Not found (job did not emit them or has not run):", index["missing"])


## Cleanup

In [ ]:
# Cleanup: nothing here creates schedules or endpoints; list them, delete only temporary pamap2-har-mon* models.
for page in sm.get_paginator("list_monitoring_schedules").paginate(NameContains=cfg["prefix"]):
    for x in page["MonitoringScheduleSummaries"]:
        print("WARNING: monitoring schedule exists (not deleted):", x["MonitoringScheduleName"])
for page in sm.get_paginator("list_models").paginate(NameContains=f"{cfg['prefix']}-mon"):
    for m in page["Models"]:
        if m["ModelName"].startswith(f"{cfg['prefix']}-mon"):
            print("Deleting temporary model", m["ModelName"])
            sm.delete_model(ModelName=m["ModelName"])
for kind, pager, key, field in (("processing", "list_processing_jobs", "ProcessingJobSummaries", "ProcessingJobName"),
                                ("transform", "list_transform_jobs", "TransformJobSummaries", "TransformJobName")):
    for page in sm.get_paginator(pager).paginate(StatusEquals="InProgress", NameContains=cfg["prefix"]):
        for x in page[key]:
            print("WARNING: still running", kind, "job:", x[field])
print("Endpoints:", [e["EndpointName"] for p in sm.get_paginator("list_endpoints").paginate() for e in p["Endpoints"]])
